# 02 - Files and GitHub (R)

**File:** `notebooks/02_files_and_github_r.ipynb`

**What this does:** Shows how a notebook finds, reads and writes files on disk, and how that work gets back to GitHub.

**How to run it:** Open this file in JupyterLab, check that the kernel shown in the
top-right corner says **R**, then choose *Run > Run All Cells*.

**Inputs:** `data/sample_stations.csv`

**Outputs:** a new file at `outputs/warm_stations_r.csv`

## Where am I? Finding the repo folder

A notebook runs from the folder it lives in (`notebooks/`), **not** from the top
of the repository. So `data/sample_stations.csv` would not be found, but
`../data/sample_stations.csv` would.

Rather than writing `../` everywhere, the cell below works out where the top of
the repo is once, and builds paths from there. Every notebook here uses this
same short block.

In [ ]:
# getwd() is the folder this notebook runs in.
repo <- getwd()
if (basename(repo) == "notebooks") repo <- dirname(repo)

data_dir <- file.path(repo, "data")

cat("Repo folder:", repo, "\n")
cat("Data folder:", data_dir, "\n")

## 1. What files are around me?

`list.files()` lists the contents of a folder. This is the notebook equivalent of
typing `ls` in a terminal.

In [ ]:
entries <- list.files(repo)

for (name in sort(entries)) {
  kind <- if (dir.exists(file.path(repo, name))) "folder" else "file"
  cat(sprintf("%-7s %s\n", kind, name))
}

## 2. Reading a file

`sample_stations.csv` is a small example file committed to the repo, so it is
there the moment you clone. `read.csv()` reads it into a data frame.

In [ ]:
stations <- read.csv(file.path(data_dir, "sample_stations.csv"))

cat(nrow(stations), "rows,", ncol(stations), "columns\n")
head(stations)

## 3. Doing something with it

Keep only the warmer stations. Nothing here changes the file on disk -- `stations`
is a copy held in memory.

In [ ]:
warm <- stations[stations$water_temp_c > 16, ]

cat(nrow(warm), "of", nrow(stations), "stations are warmer than 16 C\n")
warm[, c("station_id", "name", "water_temp_c")]

## 4. Writing a file

Write results to an `outputs/` folder rather than overwriting the input. Getting
into this habit means a mistake never destroys your original data.

In [ ]:
outputs <- file.path(repo, "outputs")
dir.create(outputs, showWarnings = FALSE)

destination <- file.path(outputs, "warm_stations_r.csv")
write.csv(warm, destination, row.names = FALSE)

cat("Wrote", destination, "\n")
cat(file.info(destination)$size, "bytes\n")

## 5. Getting your work back to GitHub

`system()` runs a shell command, so you can check on git without leaving Jupyter:

In [ ]:
cat(system("git status --short", intern = TRUE), sep = "\n")

The file you just created should appear with a `??` next to it, meaning git can
see it but is not yet tracking it.

To save your work back to GitHub, run these in a terminal
(*File > New > Terminal* in JupyterLab):

```bash
git add .
git commit -m "a short note about what you did"
git push
```

The full walkthrough -- forking, cloning and opening a pull request -- is in the
[main README](../README.md).

Two things worth knowing:

- **Notebooks produce noisy diffs.** A notebook stores its outputs inside the
  file, so re-running it shows up as a change even when you edited nothing.
  Running *Kernel > Restart Kernel and Clear Outputs* before committing keeps
  pull requests readable.
- **Anything in `data/` is deliberately ignored by git** (see `.gitignore`), so
  large downloads never end up in the repository.

## Done

Next: **`03_aquaview_stac_r.ipynb`**, which pulls real data off the internet.